## 3. Transform

Cruza partidos + clubes (local/visitante) + ligas.
Filtra temporadas desde 2010 y crea atributos de negocio para silver.


In [0]:
dbutils.widgets.removeAll()


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F


In [0]:
dbutils.widgets.text("catalogo", "football_dev")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")


In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")


In [0]:
def goal_diff_categoria(goal_diff):
    if goal_diff is None:
        return None
    abs_diff = abs(int(goal_diff))
    if abs_diff <= 1:
        return "Ajustado"
    elif abs_diff == 2:
        return "Normal"
    return "Goleada" 


In [0]:
goal_diff_udf = F.udf(goal_diff_categoria, StringType())


In [0]:
CLASSIC_PAIRS = [
    ("real_madrid", "barcelona"),
    ("man_united", "liverpool"),
    ("man_united", "man_city"),
    ("arsenal", "tottenham"),
    ("milan", "inter"),
    ("juventus", "inter"),
    ("bayern_munich", "dortmund"),
    ("paris_sg", "marseille"),
]


In [0]:
df_matches = spark.table(f"{catalogo}.{esquema_source}.matches")
df_clubs = spark.table(f"{catalogo}.{esquema_source}.clubs")
df_leagues = spark.table(f"{catalogo}.{esquema_source}.leagues")


In [0]:
df_matches = df_matches.dropna(how="all") \
    .filter(col("match_id").isNotNull() | col("league_id").isNotNull())

df_clubs = df_clubs.dropna(how="all") \
    .filter(col("club_id").isNotNull() | col("name").isNotNull())

df_leagues = df_leagues.dropna(how="all") \
    .filter(col("league_id").isNotNull() | col("league_ref").isNotNull())


In [0]:
df_home = df_clubs.select(
    col("name").alias("home_club_name"),
    col("club_ref").alias("home_club_ref"),
    col("city").alias("home_city")
)
df_away = df_clubs.select(
    col("name").alias("away_club_name"),
    col("club_ref").alias("away_club_ref"),
    col("city").alias("away_city")
)
df_league_sel = df_leagues.select(
    col("league_id"),
    col("name").alias("league_name"),
    col("country")
)


In [0]:
df_joined = df_matches.alias("m") \
    .join(df_home.alias("h"), col("m.home_club") == col("h.home_club_name"), "inner") \
    .join(df_away.alias("a"), col("m.away_club") == col("a.away_club_name"), "inner") \
    .join(df_league_sel.alias("l"), col("m.league_id") == col("l.league_id"), "inner")


In [0]:
df_filtered = df_joined.filter(col("season_year") >= 2010).orderBy("match_id")


In [0]:
classic_cond = None
for home_ref, away_ref in CLASSIC_PAIRS:
    pair = (
        ((col("home_club_ref") == home_ref) & (col("away_club_ref") == away_ref)) |
        ((col("home_club_ref") == away_ref) & (col("away_club_ref") == home_ref))
    )
    classic_cond = pair if classic_cond is None else (classic_cond | pair)


In [0]:
df_enriched = df_filtered.select(
    col("match_id"),
    col("season_year"),
    col("round"),
    col("match_date"),
    col("m.league_id").alias("league_id"),
    col("league_name"),
    col("country"),
    col("home_club"),
    col("away_club"),
    col("home_club_ref"),
    col("away_club_ref"),
    col("home_city"),
    col("away_city"),
    col("home_goals"),
    col("away_goals"),
    (col("home_goals") + col("away_goals")).alias("total_goals"),
    (col("home_goals") - col("away_goals")).alias("goal_diff"),
    when(col("home_goals") > col("away_goals"), lit("Local"))
        .when(col("home_goals") < col("away_goals"), lit("Visitante"))
        .otherwise(lit("Empate")).alias("result_type"),
    when((col("home_goals") + col("away_goals")) <= 2, lit("Baja"))
        .when((col("home_goals") + col("away_goals")) <= 4, lit("Media"))
        .otherwise(lit("Alta")).alias("match_intensity"),
    when(classic_cond, lit("Clasico")).otherwise(lit("Regular")).alias("is_classic"),
    (F.year(F.current_date()) - col("season_year")).alias("season_age"),
    col("ingestion_date")
).withColumn("goal_diff_category", goal_diff_udf(col("goal_diff")))


In [0]:
df_silver = df_enriched.select(
    "match_id", "season_year", "round", "match_date", "league_id",
    "league_name", "country", "home_club", "away_club",
    "home_club_ref", "away_club_ref", "home_city", "away_city",
    "home_goals", "away_goals", "total_goals", "goal_diff",
    "result_type", "goal_diff_category", "match_intensity",
    "is_classic", "season_age", "ingestion_date"
)


In [0]:
df_silver.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.matches_transformed")
